# Phase 6 — Coach Ratings

Coaches are the assignment's second subject. The dataset has **no coach
names** — only a `coachId` per team per match — so a coach is labelled by the
**team + competition** they managed most.

The coach card mirrors the player card, with one honest caveat built in:

* a FIFA-shaped **OVERALL (1–99)** — a **results** signal (points & goal
  difference per match), Bayesian-shrunk toward the mean so a short cup run
  can't top a full season, then displayed on the same FIFA curve as players.
  Squad quality and coaching are confounded in event data, so this is framed
  as *results with a confidence level*, not a pure "coaching ability" score.
* a **6-dimension tactical profile** (percentile among managers with ≥ 15
  games) describing **style, not quality**: Attacking Intent, Possession,
  Pressing, Defensive Solidity, **Width**, and **Line Height**.
* **confidence stars** from matches managed, a **style archetype**, and
  per-dimension **explainability** — exactly like the player card.

Managers with too few games (national teams, mid-season replacements) keep the
Overall + record but show no tactical profile (a 7-game sample can't be fairly
percentiled against a 38-game season).

In [1]:
import sys
from pathlib import Path
import pandas as pd

sys.path.insert(0, str(Path.cwd()))
import wyscout_lib as wl
pd.set_option("display.max_columns", 40); pd.set_option("display.width", 200)

pms = pd.read_parquet(wl.DATA / "player_match_stats.parquet")
matches = pd.read_parquet(wl.DATA / "matches_master.parquet")
teams = pd.read_parquet(wl.DATA / "teams_master.parquet")

coaches = wl.build_coach_stats(pms, matches, teams)
coaches.to_parquet(wl.DATA / "coach_ratings.parquet")
print(f"coaches: {len(coaches)}  |  with tactical profile (>=15 games): {coaches.has_profile.sum()}")
print(f"missing-coach matches excluded upstream (no coachId): 237 team-matches")

coaches: 211  |  with tactical profile (>=15 games): 105
missing-coach matches excluded upstream (no coachId): 237 team-matches


## Top coaches by Overall (results-based, FIFA-scaled)
The elite-squad managers should headline — and the confidence stars should
expose the short-sample national-team coaches sitting among them.

In [2]:
show = ["team_name", "competition_id", "matches", "wins", "draws", "losses",
        "points_per_match", "overall", "confidence_stars", "style"]
coaches.sort_values("overall", ascending=False).head(15)[show].reset_index(drop=True).round(2)

,team_name,competition_id,matches,wins,draws,losses,points_per_match,overall,confidence_stars,style
0,Manchester City,england,38,32,4,2,2.63,99,5,Possession Master
1,PSG,france,33,27,5,1,2.61,93,4,Possession Master
2,Bayern München,germany,25,21,1,3,2.56,91,4,Possession Master
3,Juventus,italy,37,29,5,3,2.49,91,5,Possession Master
4,Napoli,italy,35,26,7,2,2.43,90,5,Possession Master
5,Barcelona,spain,35,26,8,1,2.46,90,5,Possession Master
6,Manchester United,england,36,24,5,7,2.14,89,5,Possession Master
7,France,world_cup,14,11,3,0,2.57,89,3,—
8,Monaco,france,33,21,6,6,2.09,88,4,Wing-Overload
9,Atlético Madrid,spain,36,22,9,5,2.08,88,5,Defensive Organizer


## Sanity — do the styles match what we know about these teams?
Guardiola's City = elite possession + attack; Klopp's Liverpool & Pochettino's
Spurs = **Gegenpress** (high press, high line); Simeone's Atlético = **Defensive
Organizer** (deep block, fewest conceded); Barça play narrow (low **Width**).

In [3]:
prof = coaches[coaches.has_profile].copy()
dims = ["attacking_intent", "possession_control", "pressing_intensity",
        "defensive_solidity", "width", "line_height"]
watch = ["Manchester City", "Liverpool", "Tottenham Hotspur", "Atlético Madrid",
         "Barcelona", "Juventus", "Napoli"]
prof[prof.team_name.isin(watch)].sort_values("overall", ascending=False)[
    ["team_name", "overall"] + dims + ["style"]].reset_index(drop=True)

,team_name,overall,attacking_intent,possession_control,pressing_intensity,defensive_solidity,width,line_height,style
0,Manchester City,99,97.0,99.0,48.0,96.0,77.0,94.0,Possession Master
1,Juventus,91,78.0,86.0,34.0,98.0,43.0,90.0,Possession Master
2,Barcelona,90,81.0,94.0,36.0,92.0,1.0,66.0,Possession Master
3,Napoli,90,96.0,96.0,25.0,95.0,35.0,98.0,Possession Master
4,Atlético Madrid,88,23.0,42.0,44.0,99.0,9.0,42.0,Defensive Organizer
5,Liverpool,87,96.0,93.0,68.0,80.0,70.0,88.0,Gegenpress
6,Tottenham Hotspur,87,98.0,95.0,95.0,91.0,87.0,86.0,Gegenpress


## Style distribution
No single bucket should swallow everyone; "Balanced Approach" is the honest
fallback for managers without a pronounced tilt.

In [4]:
print(prof["style"].value_counts().to_string())

style
Balanced Approach          34
Possession Master          17
Defensive Organizer        10
Aggressive Press           10
Gegenpress                 10
Possession-Oriented         8
Wing-Overload               6
Deep Block                  5
High-Press Architect        3
Direct / Counter-Attack     2


## Possession leaders (should be City, PSG, Bayern, Napoli, Barça…)

In [5]:
prof.nlargest(8, "possession_control")[
    ["team_name", "competition_id", "possession_control", "attacking_intent",
     "line_height", "style"]].reset_index(drop=True)

,team_name,competition_id,possession_control,attacking_intent,line_height,style
0,Manchester City,england,99.0,97.0,94.0,Possession Master
1,PSG,france,98.0,92.0,56.0,Possession Master
2,Bayern München,germany,97.0,91.0,97.0,Possession Master
3,Napoli,italy,96.0,96.0,98.0,Possession Master
4,Tottenham Hotspur,england,95.0,98.0,86.0,Gegenpress
5,Barcelona,spain,94.0,81.0,66.0,Possession Master
6,Liverpool,england,93.0,96.0,88.0,Gegenpress
7,Arsenal,england,92.0,85.0,29.0,Possession Master


## The two new dimensions
**Width** (crossing / wing focus) — wing-reliant sides top it, tiki-taka sides
sit at the bottom. **Line Height** (avg x of defensive actions) — high-line
pressing sides top it, deep blocks sit low.

In [6]:
print("— Highest WIDTH (most wing/crossing-reliant) —")
print(prof.nlargest(6, "width")[["team_name", "width", "style"]].to_string(index=False))
print("\n— Lowest WIDTH (most central / through-the-middle) —")
print(prof.nsmallest(6, "width")[["team_name", "width", "style"]].to_string(index=False))
print("\n— Highest LINE (most aggressive high line) —")
print(prof.nlargest(6, "line_height")[["team_name", "line_height", "style"]].to_string(index=False))
print("\n— Lowest LINE (deepest block) —")
print(prof.nsmallest(6, "line_height")[["team_name", "line_height", "style"]].to_string(index=False))

— Highest WIDTH (most wing/crossing-reliant) —
     team_name  width               style
   Real Madrid   99.0   Possession Master
         Eibar   98.0 Possession-Oriented
Internazionale   97.0   Possession Master
          Roma   96.0          Gegenpress
 Real Sociedad   95.0   Possession Master
      Toulouse   94.0       Wing-Overload

— Lowest WIDTH (most central / through-the-middle) —
          team_name  width               style
          Barcelona    1.0   Possession Master
            Crotone    2.0   Balanced Approach
 Olympique Lyonnais    3.0   Possession Master
               Nice    4.0 Possession-Oriented
            Bologna    5.0   Balanced Approach
Borussia M'gladbach    6.0 Possession-Oriented

— Highest LINE (most aggressive high line) —
      team_name  line_height               style
          Eibar         99.0 Possession-Oriented
         Napoli         98.0   Possession Master
 Bayern München         97.0   Possession Master
           Roma         96.0      

## Most defensively solid (fewest conceded per match)

In [7]:
prof.nlargest(8, "defensive_solidity")[
    ["team_name", "competition_id", "conceded_per_match", "defensive_solidity",
     "line_height", "style"]].round(2).reset_index(drop=True)

,team_name,competition_id,conceded_per_match,defensive_solidity,line_height,style
0,Atlético Madrid,spain,0.58,99.0,42.0,Defensive Organizer
1,Juventus,italy,0.65,98.0,90.0,Possession Master
2,Manchester United,england,0.69,97.0,41.0,Possession Master
3,Manchester City,england,0.71,96.0,94.0,Possession Master
4,Napoli,italy,0.71,95.0,98.0,Possession Master
5,PSG,france,0.73,94.0,56.0,Possession Master
6,Roma,italy,0.74,93.0,96.0,Gegenpress
7,Barcelona,spain,0.77,92.0,66.0,Possession Master


## National-team coaches — Overall + record, no tactical profile
A short sample still earns an Overall (results-based, shrunk), but the low
confidence stars and the missing profile are honest about how little data
backs it. Deschamps' France (World Cup winners) should sit high.

In [8]:
np_ = coaches[(coaches.competition_id == "world_cup") & (~coaches.has_profile)]
np_.nlargest(6, "overall")[
    ["team_name", "matches", "wins", "draws", "losses", "gf", "ga",
     "overall", "confidence_stars", "has_profile"]].reset_index(drop=True)

,team_name,matches,wins,draws,losses,gf,ga,overall,confidence_stars,has_profile
0,France,14,11,3,0,27,10,89,3,False
1,Belgium,6,5,0,1,15,6,85,2,False
2,Brazil,5,3,1,1,8,3,82,2,False
3,Uruguay,4,3,0,1,5,2,81,1,False
4,Croatia,7,3,3,1,12,8,80,2,False
5,Sweden,5,3,0,2,6,4,79,2,False


Phase 6 complete → `coach_ratings.parquet`. Next: export everything to
`ratings.json` for the web app.